# simpleSpectrum — single-aperture spectrum and line IDs

Notebook version of `simpleSpectrum.m`, using `slice_lib.py` (python_slice_10222026).
Each Matlab `%%` cell is a section here:

1. Extract a channel 1 spectrum in an R = 0.5" aperture and catalog its peaks (`r_ch1`)
2. Identify the lines assuming v = 100 km/s
3. Plot the spectrum with the line identifications
4. Push deeper (S/N = 2), cull the bogus lines, re-identify and re-plot

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('.'))   # slice_lib.py lives next to this notebook

import matplotlib.pyplot as plt

from slice_lib import cat_peaks, line_id, plot_peaks, cull_lines
# slice_lib selects the non-interactive Agg backend on import, so switch to inline after it
%matplotlib inline

## Configuration
Matlab ran in `JWST/` and wrote `r_ch1*` there; here the products go to `simple_spectrum_output/`.

In [ ]:
DATA_DIR = os.path.abspath(os.path.join('..', 'JWST'))
OUT_DIR = os.path.abspath('simple_spectrum_output')
os.makedirs(OUT_DIR, exist_ok=True)

CUBE = os.path.join(DATA_DIR, 'M82_sky_v2_ch1-shortmediumlong_s3d.fits')
OUT = os.path.join(OUT_DIR, 'r_ch1')
APER = ('09:55:51.276', '69:40:44.09', 0.5)   # RA, Dec, radius (arcsec)
VLSR = 100                                    # km/s

# An edited peaklist left over from a previous run of section 4 would take
# precedence over the fresh peaklist in line_id/plot_peaks, so clear it
if os.path.exists(OUT + '-peaklist-ed.txt'):
    os.remove(OUT + '-peaklist-ed.txt')

## 1. Spectrum in an R = 0.5" aperture
Writes `r_ch1-peaklist.txt`, `r_ch1_spec.txt`, `r_ch1_cont.txt`, `r_ch1_model.txt` and `r_ch1.png`.

In [ ]:
q = cat_peaks(CUBE, OUT, APER)
plt.close('all')   # cat_peaks also saves its own spectrum figure to r_ch1.png

## 2. Identify the lines assuming v = 100 km/s

In [ ]:
line_id(OUT, VLSR)

## 3. Display the lines with their likely identity

In [ ]:
plot_peaks(OUT);

## 4. Push deeper to get more lines, some of them bogus
Re-extract at S/N = 2, then remove lines 22–24 and 27–31, which seem bogus.
The model fit is the thin black line.

In [ ]:
q = cat_peaks(CUBE, OUT, APER, 2)
plt.close('all')
cull_lines(OUT, [22, 23, 24, 27, 28, 29, 30, 31])   # 1-based, as in Matlab
line_id(OUT, VLSR)
plot_peaks(OUT);